# 面试题：MoE 如何从零实现，路由塌缩、容量溢出和负载均衡怎么处理？

## 面试回答主线

稀疏 MoE 用 router 为每个 token 计算 expert 概率，只执行 top-k expert，再按 gate 权重合并输出；因此参数量可以增长，而单 token 计算量只随 k 增长。硬 top-k 不可导，任务梯度主要通过被选择 expert 和 gate 概率传播，路由容易因早期优势塌缩到少数 expert。常用辅助损失同时约束 soft importance 与实际 load，容量因子则限制每个 expert 接收的 token 数。超过容量的 token 需要第二选择、丢弃或共享 fallback expert，三者分别影响质量、通信与计算。生产实现还必须处理跨卡 all-to-all、expert parallel 和确定性路由。

## 真实案例：多领域客服意图分类

十八条脱敏工单覆盖资金、物流和账号三个领域。每条文本被转换为六个可解释业务特征；教学目标不是替代真实编码器，而是观察同一批 token 在无均衡与有均衡时如何分配给三个 expert。

In [1]:
import math  # 导入平方根以汇总梯度范数。
from collections import Counter  # 导入计数器以统计 expert 负载。
import torch  # 导入 PyTorch 以手写 router、expert 和真实反向传播。
from torch import nn  # 导入基础模块、参数和 ModuleList 容器。
import torch.nn.functional as F  # 导入 softmax 与交叉熵等基础算子。
torch.set_num_threads(1)  # 小张量教学实验固定单线程以快速复现。
records = [  # 构造资金、物流和账号三领域的客服工单。
    ("退款三天仍未到账", [1.0, 0.0, 0.0, 1.0, 0.8, 0.4], 0),  # 资金类且具有负向状态。
    ("信用卡重复扣款", [1.0, 0.0, 0.0, 1.0, 0.9, 0.3], 0),  # 高紧急度资金风险。
    ("发票金额填写错误", [0.9, 0.0, 0.0, 0.7, 0.4, 0.5], 0),  # 发票属于资金域。
    ("优惠券抵扣失败", [0.8, 0.0, 0.0, 0.8, 0.5, 0.3], 0),  # 营销支付问题映射到资金域。
    ("支付后订单被取消", [0.9, 0.2, 0.0, 0.9, 0.8, 0.5], 0),  # 资金信号占主导的跨域工单。
    ("申请开具电子发票", [0.8, 0.0, 0.0, 0.1, 0.2, 0.4], 0),  # 非紧急资金请求。
    ("包裹超过承诺时间未送达", [0.0, 1.0, 0.0, 1.0, 0.8, 0.7], 1),  # 物流延迟工单。
    ("物流轨迹两天没更新", [0.0, 0.9, 0.0, 0.8, 0.6, 0.6], 1),  # 轨迹异常属于物流域。
    ("收货地址需要修改", [0.0, 0.8, 0.0, 0.2, 0.3, 0.4], 1),  # 地址变更属于物流域。
    ("包裹显示签收但没收到", [0.0, 1.0, 0.0, 1.0, 0.9, 0.6], 1),  # 高风险虚假签收。
    ("如何预约上门取件", [0.0, 0.8, 0.0, 0.1, 0.2, 0.4], 1),  # 普通逆向物流请求。
    ("冷链包裹温度异常", [0.1, 1.0, 0.0, 0.8, 0.9, 0.5], 1),  # 冷链异常以物流为主。
    ("账号突然无法登录", [0.0, 0.0, 1.0, 1.0, 0.8, 0.4], 2),  # 登录失败属于账号域。
    ("验证码一直收不到", [0.0, 0.0, 0.9, 0.9, 0.7, 0.4], 2),  # 验证码问题属于账号域。
    ("手机号已经更换", [0.0, 0.0, 0.8, 0.2, 0.3, 0.3], 2),  # 账号资料变更。
    ("怀疑账号被别人登录", [0.0, 0.0, 1.0, 1.0, 1.0, 0.5], 2),  # 高紧急度账号安全问题。
    ("申请永久删除账号", [0.0, 0.0, 0.9, 0.3, 0.6, 0.4], 2),  # 账号注销请求。
    ("实名认证信息错误", [0.1, 0.0, 0.9, 0.7, 0.6, 0.5], 2),  # 身份信息问题以账号域为主。
]  # 结束客服工单列表。
domain_names = ["资金", "物流", "账号"]  # 定义类别与 expert 的可读名称。
print("工单                         特征[资金,物流,账号,负向,紧急,长度]   标签")  # 输出真实案例输入表标题。
for text, features, label in records:  # 逐条展示文本、可解释特征和领域标签。
    print(f"{text:<26} {str(features):<39} {domain_names[label]}")  # 输出一条完整教学记录。

工单                         特征[资金,物流,账号,负向,紧急,长度]   标签
退款三天仍未到账                   [1.0, 0.0, 0.0, 1.0, 0.8, 0.4]          资金
信用卡重复扣款                    [1.0, 0.0, 0.0, 1.0, 0.9, 0.3]          资金
发票金额填写错误                   [0.9, 0.0, 0.0, 0.7, 0.4, 0.5]          资金
优惠券抵扣失败                    [0.8, 0.0, 0.0, 0.8, 0.5, 0.3]          资金
支付后订单被取消                   [0.9, 0.2, 0.0, 0.9, 0.8, 0.5]          资金
申请开具电子发票                   [0.8, 0.0, 0.0, 0.1, 0.2, 0.4]          资金
包裹超过承诺时间未送达                [0.0, 1.0, 0.0, 1.0, 0.8, 0.7]          物流
物流轨迹两天没更新                  [0.0, 0.9, 0.0, 0.8, 0.6, 0.6]          物流
收货地址需要修改                   [0.0, 0.8, 0.0, 0.2, 0.3, 0.4]          物流
包裹显示签收但没收到                 [0.0, 1.0, 0.0, 1.0, 0.9, 0.6]          物流
如何预约上门取件                   [0.0, 0.8, 0.0, 0.1, 0.2, 0.4]          物流
冷链包裹温度异常                   [0.1, 1.0, 0.0, 0.8, 0.9, 0.5]          物流
账号突然无法登录                   [0.0, 0.0, 1.0, 1.0, 0.8, 0.4]          账号
验证码一直收不到                   [0.0, 0.0

## Baseline（基线）：偏置 router 把所有 token 送给 expert 0

若 router 初始 bias 过大且没有均衡目标，一个通用 expert 可以独自拟合小数据，任务 loss 不会主动纠正塌缩。先把这个偏置设置显式保留，后面与同初始化的辅助损失实验比较。

In [2]:
features = torch.tensor([features for _, features, _ in records], dtype=torch.float32)  # 把业务特征转换为批量输入张量。
labels = torch.tensor([label for _, _, label in records], dtype=torch.long)  # 把三领域标签转换为整数监督张量。
class TinyExpert(nn.Module):  # 定义每个路由分支内部的两层前馈 expert。
    def __init__(self, input_dim, hidden_dim, class_count):  # 初始化扩展层和分类输出层。
        super().__init__()  # 注册基础模块状态。
        self.first_weight = nn.Parameter(torch.randn(input_dim, hidden_dim) * 0.12)  # 创建输入到隐藏层参数。
        self.first_bias = nn.Parameter(torch.zeros(hidden_dim))  # 创建 expert 隐藏层偏置。
        self.second_weight = nn.Parameter(torch.randn(hidden_dim, class_count) * 0.12)  # 创建隐藏到类别 logits 参数。
        self.second_bias = nn.Parameter(torch.zeros(class_count))  # 创建 expert 输出偏置。
    def forward(self, inputs):  # 执行两层 expert 前向传播。
        hidden = torch.tanh(inputs @ self.first_weight + self.first_bias)  # 用 tanh 得到非线性领域特征。
        return hidden @ self.second_weight + self.second_bias  # 返回当前 expert 的三类 logits。
class SparseTopOneMoE(nn.Module):  # 定义包含 router 与三个稀疏 expert 的最小 MoE 分类器。
    def __init__(self, input_dim, hidden_dim, class_count, expert_count):  # 初始化 router 参数和 expert 列表。
        super().__init__()  # 注册基础模块状态。
        self.expert_count = expert_count  # 保存 expert 数量供负载统计。
        self.router_weight = nn.Parameter(torch.randn(input_dim, expert_count) * 0.03)  # 创建输入到 expert logits 的路由权重。
        self.router_bias = nn.Parameter(torch.tensor([3.0, 0.0, 0.0]))  # 故意给第零 expert 强初始优势以复现塌缩。
        self.experts = nn.ModuleList([TinyExpert(input_dim, hidden_dim, class_count) for _ in range(expert_count)])  # 创建三个彼此独立的专家网络。
    def forward(self, inputs):  # 执行 soft router、hard top-1 dispatch 和 gate 加权输出。
        router_logits = inputs @ self.router_weight + self.router_bias  # 为每个 token 计算三个 expert 路由分数。
        router_probabilities = torch.softmax(router_logits, dim=1)  # 把路由分数转换为 soft importance。
        routes = router_probabilities.argmax(dim=1)  # 为每个 token 选择 top-1 expert 编号。
        token_outputs = []  # 创建列表按原 token 顺序收集稀疏 expert 输出。
        for token_index in range(inputs.shape[0]):  # 逐 token 执行真正被选择的唯一 expert。
            expert_index = int(routes[token_index])  # 读取当前 token 的 hard route。
            expert_logits = self.experts[expert_index](inputs[token_index : token_index + 1])[0]  # 只运行当前被选 expert 的 forward。
            gate = router_probabilities[token_index, expert_index]  # 取得被选 expert 的可导 gate 概率。
            token_outputs.append(expert_logits * gate)  # 用 gate 加权 expert 输出并保留 router 任务梯度。
        outputs = torch.stack(token_outputs, dim=0)  # 恢复原 batch 顺序的三类 logits 张量。
        importance = router_probabilities.mean(dim=0)  # 计算每个 expert 获得的平均 soft 概率质量。
        hard_load = F.one_hot(routes, num_classes=self.expert_count).to(torch.float32).mean(dim=0)  # 计算每个 expert 的实际 hard token 比例。
        balance_loss = self.expert_count * torch.sum(importance * hard_load.detach())  # 用 Switch 风格 importance-load 乘积惩罚塌缩。
        return outputs, router_probabilities, routes, balance_loss  # 返回分类 logits、路由状态和辅助损失。
torch.manual_seed(113)  # 固定基线模型参数初始化。
collapsed_model = SparseTopOneMoE(6, 10, 3, 3)  # 创建具有 expert0 强偏置的三专家模型。
with torch.no_grad():  # 初始路由检查不需要梯度图。
    initial_outputs, initial_probabilities, initial_routes, initial_balance = collapsed_model(features)  # 执行训练前完整稀疏前向。
initial_load = torch.bincount(initial_routes, minlength=3).tolist()  # 统计三个 expert 初始接收的 token 数。
print("初始 router 平均概率：", [round(float(value), 3) for value in initial_probabilities.mean(dim=0)])  # 输出 soft importance 分布。
print("初始 hard load：", dict(zip(domain_names, initial_load)))  # 输出强偏置造成的实际路由塌缩。
print("初始 balance loss：", round(float(initial_balance), 4))  # 输出辅助项在塌缩状态下的真实数值。

初始 router 平均概率： [0.911, 0.045, 0.044]
初始 hard load： {'资金': 18, '物流': 0, '账号': 0}
初始 balance loss： 2.7322


## 核心实现：同初始化分别训练 task-only 与 task + balance

两个模型拥有完全相同的初始参数。task-only 只优化分类交叉熵；balanced 模型额外优化 router importance 与 hard load 的乘积。训练循环真实执行稀疏 expert `forward`、反向传播和手动 SGD。

In [3]:
def train_moe(balance_weight):  # 训练一个辅助损失权重固定的稀疏 MoE。
    torch.manual_seed(113)  # 让不同实验从完全相同的参数与 router 偏置开始。
    model = SparseTopOneMoE(6, 10, 3, 3)  # 创建独立的三 expert 分类器。
    trace = []  # 保存代表轮次的任务 loss、辅助 loss、准确率和 load。
    for step in range(901):  # 执行真实的稀疏前向、反向与参数更新。
        logits, probabilities, routes, balance_loss = model(features)  # 计算分类输出和路由状态。
        task_loss = F.cross_entropy(logits, labels)  # 计算领域分类交叉熵。
        total_loss = task_loss + balance_weight * balance_loss  # 根据实验设置加入负载均衡辅助目标。
        total_loss.backward()  # 对 router 和被选 expert 执行真实反向传播。
        with torch.no_grad():  # 手动更新不应被 autograd 记录。
            for parameter in model.parameters():  # 遍历 router 与三个 expert 的全部参数。
                if parameter.grad is not None:  # 未被任何 token 选择的 expert 本轮可能没有梯度。
                    parameter -= 0.08 * parameter.grad  # 使用固定学习率执行 SGD 更新。
                    parameter.grad.zero_()  # 清空当前参数梯度避免跨轮累积。
        if step in {0, 20, 100, 300, 600, 900}:  # 记录能够说明路由演化的代表轮次。
            predictions = logits.detach().argmax(dim=1)  # 取得当前三类预测。
            accuracy = float((predictions == labels).to(torch.float32).mean())  # 计算当前分类准确率。
            load = torch.bincount(routes.detach(), minlength=3).tolist()  # 统计每个 expert 的 hard token 数。
            trace.append((step, float(task_loss.detach()), float(balance_loss.detach()), accuracy, load))  # 保存训练状态。
    return model, trace  # 返回训练后的模型和完整代表轨迹。
task_only_model, task_only_trace = train_moe(0.0)  # 训练不含任何负载均衡的塌缩对照。
balanced_model, balanced_trace = train_moe(0.8)  # 训练加入较强均衡约束的 MoE。
print("轮次 | task-only: task/balance/acc/load       | balanced: task/balance/acc/load")  # 输出两种训练设置的轨迹标题。
for task_row, balanced_row in zip(task_only_trace, balanced_trace):  # 对齐相同轮次的两组状态。
    print(f"{task_row[0]:>4} | {task_row[1]:.3f}/{task_row[2]:.3f}/{task_row[3]:.0%}/{task_row[4]} | {balanced_row[1]:.3f}/{balanced_row[2]:.3f}/{balanced_row[3]:.0%}/{balanced_row[4]}")  # 输出 loss、准确率和实际负载。

轮次 | task-only: task/balance/acc/load       | balanced: task/balance/acc/load
   0 | 1.093/2.732/56%/[18, 0, 0] | 1.093/2.732/56%/[18, 0, 0]
  20 | 1.029/2.736/83%/[18, 0, 0] | 1.058/1.775/83%/[18, 0, 0]
 100 | 0.394/2.825/100%/[18, 0, 0] | 1.022/0.999/78%/[7, 8, 3]
 300 | 0.036/2.902/100%/[18, 0, 0] | 0.524/0.990/83%/[7, 6, 5]
 600 | 0.012/2.920/100%/[18, 0, 0] | 0.046/1.000/100%/[6, 6, 6]
 900 | 0.007/2.927/100%/[18, 0, 0] | 0.013/1.000/100%/[6, 6, 6]


## 结果表：逐工单 route、概率与分类

任务 loss 能让单一 expert 学会全部类别，所以 task-only 的准确率可能很高但没有稀疏扩容价值。balanced 模型的关键证据是 soft importance 与 hard load 同时展开，而不是只看最终分类分数。

In [4]:
with torch.no_grad():  # 评估阶段关闭梯度图以得到稳定路由结果。
    task_logits, task_probabilities, task_routes, task_balance = task_only_model(features)  # 计算 task-only 最终状态。
    balanced_logits, balanced_probabilities, balanced_routes, final_balance = balanced_model(features)  # 计算 balanced 最终状态。
task_predictions = task_logits.argmax(dim=1)  # 取得 task-only 三类预测。
balanced_predictions = balanced_logits.argmax(dim=1)  # 取得 balanced 三类预测。
task_accuracy = float((task_predictions == labels).to(torch.float32).mean())  # 计算 task-only 准确率。
balanced_accuracy = float((balanced_predictions == labels).to(torch.float32).mean())  # 计算 balanced 准确率。
task_load = torch.bincount(task_routes, minlength=3)  # 统计 task-only expert token 数。
balanced_load = torch.bincount(balanced_routes, minlength=3)  # 统计 balanced expert token 数。
print("工单                     标签  task路由  balanced路由  balanced预测  gate概率")  # 输出逐工单路由结果表标题。
for index, (text, _, label) in enumerate(records):  # 遍历原始工单、特征和标签。
    gate_view = [round(float(value), 3) for value in balanced_probabilities[index]]  # 格式化 balanced router 的三个概率。
    print(f"{text:<24} {domain_names[label]:<4} {int(task_routes[index]):>8} {int(balanced_routes[index]):>13} {domain_names[int(balanced_predictions[index])]:>12}  {gate_view}")  # 输出路由、预测和 soft gate。
print("task-only load：", task_load.tolist(), f"，准确率={task_accuracy:.1%}")  # 输出塌缩模型负载与任务效果。
print("balanced load：", balanced_load.tolist(), f"，准确率={balanced_accuracy:.1%}")  # 输出均衡模型负载与任务效果。

工单                     标签  task路由  balanced路由  balanced预测  gate概率
退款三天仍未到账                 资金          0             2           资金  [0.063, 0.142, 0.794]
信用卡重复扣款                  资金          0             2           资金  [0.063, 0.141, 0.796]
发票金额填写错误                 资金          0             2           资金  [0.121, 0.15, 0.729]
优惠券抵扣失败                  资金          0             2           资金  [0.142, 0.167, 0.691]
支付后订单被取消                 资金          0             2           资金  [0.073, 0.239, 0.688]
申请开具电子发票                 资金          0             2           资金  [0.269, 0.135, 0.596]
包裹超过承诺时间未送达              物流          0             1           物流  [0.057, 0.875, 0.068]
物流轨迹两天没更新                物流          0             1           物流  [0.104, 0.811, 0.085]
收货地址需要修改                 物流          0             1           物流  [0.278, 0.626, 0.096]
包裹显示签收但没收到               物流          0             1           物流  [0.058, 0.874, 0.068]
如何预约上门取件                 物流          0       

## 结果解读

task-only 允许 expert0 变成一个普通 dense 网络，分类 loss 可以下降，却浪费另外两个 expert 的容量。辅助损失通过 soft router 概率获得梯度，逐步打破早期偏置并让 hard route 展开。均衡不是越平均越好：真实领域分布可能不均，过大系数会为了平均负载牺牲任务路由，因此要同时看质量、负载、跨卡通信和 overflow。

## 失败案例：热门领域突发导致单 expert 超容量

即使全局训练负载均衡，一个 batch 也可能全是退款。容量设为 `ceil(tokens / experts)` 时，top-1 直接 dispatch 会丢弃溢出 token。修复策略先尝试 top-2 的第二 expert，仍无槽位则送共享 fallback；下面打印真实分配账本。

In [5]:
burst_features = features[:9].clone()  # 取九条以前六条资金为主的特征构造热点 batch。
burst_features[:6, 0] += 0.5  # 进一步强化前六条退款与支付信号。
with torch.no_grad():  # 容量调度演示不需要梯度图。
    burst_router_logits = burst_features @ balanced_model.router_weight + balanced_model.router_bias  # 计算热点 batch 的 router logits。
    burst_router_probabilities = torch.softmax(burst_router_logits, dim=1)  # 得到每 token 的三个 expert 概率。
    top_two_probabilities, top_two_experts = torch.topk(burst_router_probabilities, k=2, dim=1)  # 取得每 token 的首选与第二选择。
capacity = math.ceil(len(burst_features) / 3)  # 使用 capacity factor 一得到每 expert 三个槽位。
top_one_assignments = top_two_experts[:, 0]  # 取得不考虑容量的 top-1 路由。
top_one_load = torch.bincount(top_one_assignments, minlength=3)  # 统计热点 batch 的原始负载。
top_one_dropped = int(sum(max(0, int(load) - capacity) for load in top_one_load))  # 计算直接 top-1 dispatch 的溢出 token 数。
remaining = [capacity, capacity, capacity]  # 初始化三个 expert 的可用容量槽位。
final_assignments = []  # 保存首选、第二选择或 fallback 的逐 token 分配。
for token_index in range(len(burst_features)):  # 按原 batch 顺序执行确定性容量调度。
    first = int(top_two_experts[token_index, 0])  # 读取当前 token 的 top-1 expert。
    second = int(top_two_experts[token_index, 1])  # 读取当前 token 的 top-2 expert。
    if remaining[first] > 0:  # 首选 expert 仍有空闲槽位时直接接收。
        assignment = f"expert-{first}"  # 记录首选 expert 分配。
        remaining[first] -= 1  # 消耗首选 expert 的一个容量槽位。
    elif remaining[second] > 0:  # 首选已满但第二选择仍有槽位时改道。
        assignment = f"expert-{second}(second)"  # 标记通过第二选择完成分配。
        remaining[second] -= 1  # 消耗第二 expert 的一个容量槽位。
    else:  # 两个候选 expert 都没有剩余容量。
        assignment = "shared-fallback"  # 使用始终可用但增加计算的共享 fallback expert。
    final_assignments.append(assignment)  # 保存当前 token 的最终调度结果。
print("热点 batch top-1 load：", top_one_load.tolist(), "，单 expert 容量：", capacity, "，直接丢弃：", top_one_dropped)  # 输出容量溢出基线。
for index, assignment in enumerate(final_assignments):  # 展示每个热点 token 的真实调度路径。
    print(f"token-{index}: top2={top_two_experts[index].tolist()} prob={[round(float(value), 3) for value in top_two_probabilities[index]]} -> {assignment}")  # 输出 top2、概率和最终去向。
fallback_count = final_assignments.count("shared-fallback")  # 统计共享 fallback 实际承接的 token 数。
print("修复后无 token 丢失；second-choice 数：", sum("second" in assignment for assignment in final_assignments), "，fallback 数：", fallback_count)  # 输出容量修复的成本构成。

热点 batch top-1 load： [0, 3, 6] ，单 expert 容量： 3 ，直接丢弃： 3
token-0: top2=[2, 1] prob=[0.922, 0.058] -> expert-2
token-1: top2=[2, 1] prob=[0.923, 0.058] -> expert-2
token-2: top2=[2, 1] prob=[0.895, 0.065] -> expert-2
token-3: top2=[2, 1] prob=[0.877, 0.075] -> expert-1(second)
token-4: top2=[2, 1] prob=[0.869, 0.106] -> expert-1(second)
token-5: top2=[2, 0] prob=[0.832, 0.102] -> expert-0(second)
token-6: top2=[1, 2] prob=[0.875, 0.068] -> expert-1
token-7: top2=[1, 0] prob=[0.811, 0.104] -> expert-0(second)
token-8: top2=[1, 0] prob=[0.626, 0.278] -> expert-0(second)
修复后无 token 丢失；second-choice 数： 5 ，fallback 数： 0


## 生产差距与落地清单

教学模型按 Python 循环 dispatch，真实 MoE 要把 token 按 expert 排序、跨设备 all-to-all，再恢复原顺序；还需 top-2 gate 归一化、容量因子、padding token 排除、expert dropout 与路由噪声。监控不能只看平均 load，应记录每层每 expert 的 P50/P99 token 数、overflow/fallback、router 熵、跨卡字节和分领域质量。共享 fallback 消除丢弃但会增加计算，必须纳入 SLO 与成本预算。

## 最小回归测试

断言只保护稀疏执行、负载变化、容量门禁与质量底线；路由轨迹、逐 token gate 和 overflow 账本才是主要证据。

In [6]:
assert initial_load == [len(records), 0, 0]  # 验证强 router 偏置真实复现初始 expert 塌缩。
assert task_load.max() > task_load.min() + 3  # 验证没有均衡损失时负载仍明显偏斜。
assert int(balanced_load.max() - balanced_load.min()) <= 2  # 验证辅助目标把最终 hard load 拉到近似均衡。
assert balanced_accuracy >= 0.9  # 验证均衡没有破坏教学分类任务的基本质量。
assert top_one_dropped > 0  # 固化热点 batch 在直接 top-1 容量门禁下会溢出的失败案例。
assert len(final_assignments) == len(burst_features)  # 验证第二选择与 fallback 为每个 token 给出最终去向。
assert all(remaining_expert >= 0 for remaining_expert in remaining)  # 验证修复调度从未超卖任何 expert 槽位。
print("最小回归测试通过：路由塌缩、负载均衡、容量溢出和 fallback 调度均符合预期。")  # 输出完整顺序执行成功的明确结论。

最小回归测试通过：路由塌缩、负载均衡、容量溢出和 fallback 调度均符合预期。
